# 12 — LLM-as-a-Judge Evaluation Framework

**Model:** DeepSeek-V3 via Nebius AI Studio
**Pattern:** Systematic Agent Output Evaluation
**Difficulty:** ⭐⭐⭐☆☆

---

## The Problem

How do you know if your agent is actually good?

Human evaluation is expensive and doesn't scale. Traditional NLP metrics (BLEU, ROUGE) don't capture quality for open-ended tasks. **LLM-as-a-Judge** uses a language model as an automated evaluator — scoring outputs on a structured rubric that approximates human judgment.

## Design Principles for LLM Judges

A good judge is:
1. **Rubric-based** — scores specific, well-defined criteria
2. **Position-unbiased** — doesn't favor the first option in pairwise comparisons
3. **Explanation-required** — must justify every score
4. **Calibrated** — uses the full scoring range, not just 8-10

## Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_nebius import ChatNebius
from langchain_core.messages import HumanMessage, SystemMessage
from typing import TypedDict, List, Optional, Dict
from pydantic import BaseModel, Field

judge_llm = ChatNebius(
    model="deepseek-ai/DeepSeek-V3",
    temperature=0,
    api_key=os.environ["NEBIUS_API_KEY"]
)
print("Judge LLM ready.")

## Evaluation Rubric

In [ ]:
RUBRIC = {
    "accuracy":      {"description": "Are facts correct? No hallucinations?",         "weight": 0.30},
    "relevance":     {"description": "Does it directly address the question?",         "weight": 0.25},
    "completeness":  {"description": "Does it cover all important aspects?",           "weight": 0.20},
    "clarity":       {"description": "Is it well-structured and readable?",            "weight": 0.15},
    "conciseness":   {"description": "Is it appropriately concise?",                   "weight": 0.10},
}

assert abs(sum(c["weight"] for c in RUBRIC.values()) - 1.0) < 0.001
print(f"Rubric loaded: {list(RUBRIC.keys())}")

## Pydantic Schemas

In [ ]:
class CriterionScore(BaseModel):
    score: float = Field(ge=1.0, le=10.0, description="Score from 1-10")
    reasoning: str = Field(description="One sentence explaining this score")

class EvaluationResult(BaseModel):
    accuracy: CriterionScore
    relevance: CriterionScore
    completeness: CriterionScore
    clarity: CriterionScore
    conciseness: CriterionScore
    overall_assessment: str = Field(description="2-3 sentence summary")

class PairwiseResult(BaseModel):
    winner: str = Field(description="'A', 'B', or 'tie'")
    reasoning: str
    a_strengths: str
    b_strengths: str

## Absolute Scoring

In [ ]:
ABSOLUTE_JUDGE_SYSTEM = """
You are an expert evaluator for AI-generated responses.
Score each criterion from 1.0 (very poor) to 10.0 (excellent).
Criteria: accuracy (facts correct?), relevance (addresses question?),
completeness (covers all aspects?), clarity (well-structured?), conciseness (appropriately brief?).
Be critical. Use the full range. Always justify each score.
"""

absolute_judge = judge_llm.with_structured_output(EvaluationResult)

def evaluate_response(question: str, response: str, context: str = "") -> EvaluationResult:
    context_section = f"\nContext:\n{context}" if context else ""
    return absolute_judge.invoke([
        SystemMessage(content=ABSOLUTE_JUDGE_SYSTEM),
        HumanMessage(content=(
            f"Question/Task: {question}{context_section}\n\n"
            f"Response to evaluate:\n{response}"
        ))
    ])

def compute_weighted_score(result: EvaluationResult) -> float:
    scores = {
        "accuracy": result.accuracy.score, "relevance": result.relevance.score,
        "completeness": result.completeness.score, "clarity": result.clarity.score,
        "conciseness": result.conciseness.score,
    }
    return sum(scores[k] * RUBRIC[k]["weight"] for k in scores)

def print_evaluation(result: EvaluationResult, label: str = "Response"):
    weighted = compute_weighted_score(result)
    print(f"\n{'='*60}")
    print(f"EVALUATION: {label}")
    print(f"{'='*60}")
    print(f"Weighted Score: {weighted:.1f}/10.0\n")
    for criterion in RUBRIC:
        cr = getattr(result, criterion)
        bar = "█" * int(cr.score) + "░" * (10 - int(cr.score))
        print(f"  {criterion.capitalize():<14} [{bar}] {cr.score:.1f}  — {cr.reasoning}")
    print(f"\n  Assessment: {result.overall_assessment}")

## Pairwise Comparison

In [ ]:
PAIRWISE_JUDGE_SYSTEM = """
You are an expert evaluator. Compare two AI-generated responses to the same question.
Decide which is better (A, B, or tie). Focus on content quality, not order.
"""

pairwise_judge = judge_llm.with_structured_output(PairwiseResult)

def compare_responses(question: str, response_a: str, response_b: str) -> PairwiseResult:
    return pairwise_judge.invoke([
        SystemMessage(content=PAIRWISE_JUDGE_SYSTEM),
        HumanMessage(content=(
            f"Question: {question}\n\n"
            f"Response A:\n{response_a}\n\n"
            f"Response B:\n{response_b}"
        ))
    ])

## Demo 1: Absolute Evaluation

In [ ]:
question = "What is Egypt's economic situation and main challenges in 2025?"

response_a = """
Egypt's economy faces significant headwinds in 2025. After an IMF-backed reform package,
the Egyptian pound stabilized at ~48 EGP/USD. GDP growth recovered to ~4%, supported by
remittances ($22B annually), tourism (~$13B), and Suez Canal receipts (~$9B).
Key challenges: inflation still above 25%, debt-to-GDP near 95%, youth unemployment.
"""

response_b = """
Egypt has some economic problems. The currency went down but now it's better.
There's inflation and debt. The government is trying to fix things with the IMF.
Tourism is important. Overall the situation is challenging but improving.
"""

print("Evaluating Response A (detailed)...")
result_a = evaluate_response(question, response_a)
print_evaluation(result_a, "Response A (detailed)")

print("\nEvaluating Response B (vague)...")
result_b = evaluate_response(question, response_b)
print_evaluation(result_b, "Response B (vague)")

## Demo 2: Pairwise Comparison

In [ ]:
comparison = compare_responses(question, response_a, response_b)

print(f"\n{'='*60}")
print("PAIRWISE COMPARISON RESULT")
print(f"{'='*60}")
print(f"Winner: Response {comparison.winner.upper()}")
print(f"Reasoning: {comparison.reasoning}")
print(f"\nA's strengths: {comparison.a_strengths}")
print(f"B's strengths: {comparison.b_strengths}")

## Demo 3: Batch Evaluation

In [ ]:
test_cases = [
    {
        "notebook": "01 — Self-Critique",
        "question": "Explain why LLMs hallucinate.",
        "response": (
            "LLMs hallucinate because they generate statistically probable token sequences "
            "rather than retrieving verified facts. During training, models learn correlations "
            "between concepts — but these can lead to plausible-sounding but incorrect outputs, "
            "especially for obscure topics. Without a ground-truth retrieval mechanism, "
            "the model fills gaps with high-probability continuations that may not be accurate."
        )
    },
    {
        "notebook": "05 — Multi-Agent",
        "question": "Summarize the AI chip market in 2025.",
        "response": (
            "NVIDIA dominates the AI chip market in 2025 with ~85% market share in data center GPUs, "
            "driven by H100/H200 demand and its CUDA ecosystem. AMD is the primary challenger, "
            "gaining traction in inference workloads with MI300X at a 20-30% price advantage. "
            "Intel's Gaudi 3 has struggled due to a less mature software stack."
        )
    },
]

print("Running batch evaluation...\n")
batch_results = []
for case in test_cases:
    result = evaluate_response(case["question"], case["response"])
    score = compute_weighted_score(result)
    batch_results.append({"notebook": case["notebook"], "score": score})
    print(f"  {case['notebook']}: {score:.1f}/10")

print("\n" + "="*60)
print("BATCH SUMMARY")
print("="*60)
batch_results.sort(key=lambda x: x["score"], reverse=True)
for r in batch_results:
    bar = "█" * int(r["score"]) + "░" * (10 - int(r["score"]))
    print(f"  {r['notebook']:<28} [{bar}] {r['score']:.1f}")

## Key Takeaways

| Concept | Implementation |
|---------|---------------|
| **Rubric-based scoring** | 5 independent criteria with defined weights |
| **Structured output** | Pydantic schemas prevent hallucinated/unparseable scores |
| **Reasoning required** | Every score includes a justification |
| **Pairwise vs absolute** | Pairwise is more reliable for ranking |

## Closing Thoughts

You've now implemented 12 distinct agentic architectures — from simple reflection loops to 3-tier hierarchies and automated evaluation pipelines.

**The best architecture is the simplest one that meets your requirements.** Start with a single ReAct agent. Add planning when you need multi-step goals. Add multi-agent only when specialization provides clear value. And always measure.

Happy building.